In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import ta
from hmmlearn.hmm import GaussianHMM
import matplotlib.pyplot as plt
import seaborn as sns
import vectorbt as vbt
import pyfolio as pf
import warnings

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

In [ ]:
# Téléchargement des données du CAC 40
df = yf.download("^FCHI", start="2000-01-01")
df['Returns'] = df['Close'].pct_change()
df = df.dropna()

# Séparation Train/Test (Out-of-sample)
train_set = df[df.index < '2010-01-01']
test_set = df[df.index >= '2010-01-01']

# On récupère les rentabilités pour le modèle
X_train = train_set[['Returns']].values
X_test = test_set[['Returns']].values

In [ ]:
# Initialisation et entraînement
# n_components=2 (Marché calme vs Marché volatil)
model = GaussianHMM(n_components=2, covariance_type="full", n_iter=1000, random_state=42)
model.fit(X_train)

# Prédiction des régimes sur le jeu de test
test_set['Regime'] = model.predict(X_test)

print(f"Score du modèle : {model.score(X_test)}")

In [ ]:
# Identification de l'état "Extrême" (plus forte variance)
variances = np.array([np.diag(model.covars_[i]) for i in range(model.n_components)])
extreme_regime = np.argmax(variances) # L'indice de l'état le plus volatil
normal_regime = 1 - extreme_regime

# Visualisation des régimes sur les prix
plt.figure(figsize=(15, 8))
for regime in range(model.n_components):
    mask = test_set['Regime'] == regime
    plt.scatter(test_set.index[mask], test_set['Close'][mask], label=f"Régime {regime}", s=10)
plt.title("Détection des régimes de marché sur le CAC 40 (Test Set)")
plt.legend()
plt.show()

# Comparaison des distributions des rentabilités
sns.displot(data=test_set, x="Returns", hue="Regime", kind="kde", fill=True)
plt.title("Distribution des rentabilités par régime")
plt.show()

In [ ]:
# Moyennes Mobiles Exponentielles
test_set['EMA_50'] = ta.trend.ema_indicator(test_set['Close'], window=50)
test_set['EMA_100'] = ta.trend.ema_indicator(test_set['Close'], window=100)

# MACD
macd_indicator = ta.trend.MACD(test_set['Close'], window_fast=12, window_slow=26, window_sign=9)
test_set['MACD'] = macd_indicator.macd()
test_set['MACD_Signal'] = macd_indicator.macd_signal()

In [ ]:
# Stratégie 1: EMA + HMM
test_set['Signal_EMA'] = (test_set['EMA_50'] > test_set['EMA_100']) & (test_set['Regime'] == normal_regime)

# Stratégie 2: MACD + HMM
test_set['Signal_MACD'] = (test_set['MACD'] > test_set['MACD_Signal']) & (test_set['Regime'] == normal_regime)

# Stratégie 3: HMM Uniquement (Sortie de crise)
test_set['Signal_HMM'] = (test_set['Regime'] == normal_regime)

# Benchmark: Buy & Hold
test_set['Signal_BH'] = True

In [ ]:
# Simulation des portefeuilles
pf_ema = vbt.Portfolio.from_signals(test_set['Close'], test_set['Signal_EMA'], freq="D")
pf_macd = vbt.Portfolio.from_signals(test_set['Close'], test_set['Signal_MACD'], freq="D")
pf_hmm = vbt.Portfolio.from_signals(test_set['Close'], test_set['Signal_HMM'], freq="D")
pf_bh = vbt.Portfolio.from_signals(test_set['Close'], test_set['Signal_BH'], freq="D")

# Affichage des résultats cumulés
plt.figure(figsize=(15, 8))
pf_bh.cumulative_returns().plot(label="Benchmark (Buy & Hold)")
pf_ema.cumulative_returns().plot(label="Stratégie EMA + HMM")
pf_macd.cumulative_returns().plot(label="Stratégie MACD + HMM")
pf_hmm.cumulative_returns().plot(label="Stratégie HMM Exit", linewidth=3)
plt.title("Comparaison des performances des stratégies")
plt.legend()
plt.show()

# Statistiques détaillées de la meilleure stratégie (HMM Exit)
print(pf_hmm.stats())